In [ ]:
from pathlib import Path
import SimpleITK as sitk
import numpy as np
import pandas as pd
from radiomics import featureextractor
from tqdm import tqdm


# === PERCORSI PRINCIPALI  ===

PROJECT_DIR = Path("../RadiomiK/Radiomica")
DATA_DIR = PROJECT_DIR / "Abdomen_200825_512_PyRad_ready"  # cartella che contiene i protocolli
PARAMS = PROJECT_DIR / "params_radiomics.yaml"     # file YAML dei parametri


# === FUNZIONE: carica una serie DICOM (una rep_x) in SimpleITK.Image ===

def carica_serie(cartella: Path) -> sitk.Image:
    """
    Legge tutti i DICOM in 'cartella' come una serie 3D.
    """
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(cartella))
    if not series_ids:
        raise RuntimeError(f"Nessuna serie DICOM trovata in {cartella}")

    # Supponiamo UNA sola serie per cartella rep_x
    files = reader.GetGDCMSeriesFileNames(str(cartella), series_ids[0])
    reader.SetFileNames(files)
    img = reader.Execute()
    return img


# === FUNZIONE: estrae le feature da TUTTE le ROI (etichette) di una serie ===

def estrai_roi_per_serie(img: sitk.Image,
                         mask: sitk.Image,
                         protocollo: str,
                         ripetizione: str,
                         extractor: featureextractor.RadiomicsFeatureExtractor):
    """
    img  = immagine CT (SimpleITK.Image)
    mask = maschera con etichette 1..19 (SimpleITK.Image)
    protocollo = nome del protocollo (cartella)
    ripetizione = nome della rep (es. "rep_1")
    extractor = istanza di RadiomicsFeatureExtractor già configurata

    Ritorna: lista di dizionari (una riga per ROI)
    """
    arr = sitk.GetArrayFromImage(mask)  # (z, y, x)
    labels, counts = np.unique(arr, return_counts=True)

    # Filtra background (0)
    mask_nonzero = labels != 0
    labels = labels[mask_nonzero]
    counts = counts[mask_nonzero]

    if len(labels) == 0:
        print("    [ATTENZIONE] Nessuna ROI trovata in questa maschera.")
        return []

    # (Opzionale) Warning se qualche ROI è troppo piccola
    for lab, cnt in zip(labels, counts):
        if cnt < 20:
            print(f"    [WARNING] ROI label {lab} ha solo {cnt} voxel (molto piccola).")

    risultati = []

    for label in tqdm(labels, desc=f"    ROI extraction", leave=False):
        # Imposta l'etichetta corrente
        extractor.settings["label"] = int(label)

        # Esegui l'estrazione
        feats = extractor.execute(img, mask)

        # Prendi solo le feature radiomiche (quelle che iniziano con "original_")
        row = {
            k: float(v)
            for k, v in feats.items()
            if k.startswith("original_")
        }

        # Metadati per l'analisi
        row["protocol"] = protocollo
        row["repeat"] = ripetizione
        row["roi"] = int(label)

        risultati.append(row)

    return risultati


# === MAIN ===

def main():
    if not PARAMS.exists():
        raise FileNotFoundError(f"File parametri non trovato: {PARAMS}")

    print(f"Carico parametri da: {PARAMS}")
    extractor = featureextractor.RadiomicsFeatureExtractor(str(PARAMS))

    tutte_le_righe = []

    # Ogni sottocartella di DATA_DIR è un protocollo
    protocolli = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())

    # Conta totale delle ripetizioni
    totale_ripetizioni = sum(
        len([p for p in prot_dir.iterdir() if p.is_dir() and p.name.startswith("rep_")])
        for prot_dir in protocolli
    )

    print(f"\nTrovati {len(protocolli)} protocolli con {totale_ripetizioni} ripetizioni totali\n")

    # Barra di avanzamento principale per tutte le ripetizioni
    with tqdm(total=totale_ripetizioni, desc="Processamento totale", unit="rep") as pbar_main:
        for prot_dir in protocolli:
            protocol_name = prot_dir.name
            
            # Ogni sottocartella rep_x è una ripetizione
            rep_dirs = sorted(p for p in prot_dir.iterdir()
                              if p.is_dir() and p.name.startswith("rep_"))

            for rep_dir in rep_dirs:
                rep_name = rep_dir.name
                pbar_main.set_description(f"Processing {protocol_name}/{rep_name}")

                # Controllo maschera
                mask_path = rep_dir / "roi.nrrd"
                if not mask_path.exists():
                    tqdm.write(f"  [ATTENZIONE] {protocol_name}/{rep_name}: ROI non trovata, salto.")
                    pbar_main.update(1)
                    continue

                # Carica immagine e maschera
                try:
                    img = carica_serie(rep_dir)
                except Exception as e:
                    tqdm.write(f"  [ERRORE] {protocol_name}/{rep_name}: Lettura DICOM fallita: {e}")
                    pbar_main.update(1)
                    continue

                mask = sitk.ReadImage(str(mask_path))

                # Estrai feature per tutte le ROI di questa serie
                righe = estrai_roi_per_serie(img, mask, protocol_name, rep_name, extractor)
                tutte_le_righe.extend(righe)
                
                pbar_main.update(1)

    # Converte in DataFrame e salva
    if not tutte_le_righe:
        print("\n[ATTENZIONE] Nessuna feature estratta. Controlla maschere e DICOM.")
        return

    df = pd.DataFrame(tutte_le_righe)
    out_csv = PROJECT_DIR / "ABDOMEN_radiomics_features.csv"
    df.to_csv(out_csv, index=False)
    print(f"\n✓ FATTO! {len(tutte_le_righe)} righe di feature salvate in: {out_csv}")


if __name__ == "__main__":
    main()


Carico parametri da: /Users/irenetenerani/Desktop/RadiomiK/Radiomica/params_radiomics.yaml

Trovati 18 protocolli con 90 ripetizioni totali



Processing Abdomen_HD_1,00_Br32_A0/rep_1:   0%|          | 0/90 [00:00<?, ?rep/s]GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 ne


✓ FATTO! 1710 righe di feature salvate in: /Users/irenetenerani/Desktop/RadiomiK/Radiomica/ABDOMEN_radiomics_features.csv


In [ ]:
#Mostra le prime 10 righe del file CSV delle features estratte
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("../RadiomiK/Radiomica")
csv_path = PROJECT_DIR / "Abdomen_radiomics_features.csv"

if not csv_path.exists():
    print(f"File non trovato: {csv_path}")
else:
    df = pd.read_csv(csv_path)
    
    print(f"Dataset: {len(df)} righe totali, {len(df.columns)} colonne\n")
    
    # Mostra informazioni sui metadati
    print("=== Protocolli presenti ===")
    print(df['protocol'].value_counts().sort_index())
    print(f"\n=== ROI presenti ===")
    print(sorted(df['roi'].unique()))
    print(f"\n=== Ripetizioni per protocollo ===")
    print(df['repeat'].value_counts().sort_index())
    
    # Mostra le prime 10 righe con focus sui metadati e alcune feature
    print("\n=== Prime 10 righe (metadati + alcune features) ===")
    
    # Colonne metadati
    metadata_cols = ['protocol', 'repeat', 'roi']
    
    # Prendi alcune feature radiomiche di esempio
    feature_cols = [col for col in df.columns if col.startswith('original_')][:5]
    
    cols_to_show = metadata_cols + feature_cols
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    
    print(df[cols_to_show].head(10))
    
    print(f"\n(Mostrate {len(feature_cols)} delle {len(feature_cols) + len(metadata_cols) - len(feature_cols)} feature totali)")


Dataset: 1710 righe totali, 96 colonne

=== Protocolli presenti ===
protocol
Abdomen_HD_1,00_Br32_A0    95
Abdomen_HD_1,00_Br32_A3    95
Abdomen_HD_1,00_Br36_A0    95
Abdomen_HD_1,00_Br36_A3    95
Abdomen_HD_1,00_Br40_A0    95
Abdomen_HD_1,00_Br40_A3    95
Abdomen_LD_1,00_Br32_A0    95
Abdomen_LD_1,00_Br32_A3    95
Abdomen_LD_1,00_Br36_A0    95
Abdomen_LD_1,00_Br36_A3    95
Abdomen_LD_1,00_Br40_A0    95
Abdomen_LD_1,00_Br40_A3    95
Abdomen_ST_1,00_Br32_A0    95
Abdomen_ST_1,00_Br32_A3    95
Abdomen_ST_1,00_Br36_A0    95
Abdomen_ST_1,00_Br36_A3    95
Abdomen_ST_1,00_Br40_A0    95
Abdomen_ST_1,00_Br40_A3    95
Name: count, dtype: int64

=== ROI presenti ===
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

=== Ripetizioni per protocollo ===
repeat
rep_1    342
rep_2    342
rep_3    342
rep_4    342
rep_5    342
Name: count, dtype: int64

=== Prime 10 righe (metadati + alcune features) ===
                  protocol repeat  roi  original_firstorder_10Percentile  \
0  A

In [ ]:
# Visualizza tutte le feature di un protocollo e ripetizione specifici
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("../RadiomiK/Radiomica")
csv_path = PROJECT_DIR / "Abdomen_radiomics_features.csv"

# === MODIFICA QUI PER SELEZIONARE PROTOCOLLO E RIPETIZIONE ===
PROTOCOLLO_SCELTO = "Abdomen_HD_HR_1.00_Bl60_A0"  # Cambia con il protocollo
RIPETIZIONE_SCELTA = "rep_1"  # Cambia con la ripetizione
ROI_SCELTA = None  # un numero (es. 1) per vedere solo una ROI, oppure None per tutte

if not csv_path.exists():
    print(f"File non trovato: {csv_path}")
else:
    df = pd.read_csv(csv_path)
    
    # Filtra per protocollo e ripetizione
    df_filtrato = df[(df['protocol'] == PROTOCOLLO_SCELTO) & (df['repeat'] == RIPETIZIONE_SCELTA)]
    
    # Filtra anche per ROI se specificata
    if ROI_SCELTA is not None:
        df_filtrato = df_filtrato[df_filtrato['roi'] == ROI_SCELTA]
    
    if df_filtrato.empty:
        print(f"Nessun dato trovato per {PROTOCOLLO_SCELTO} / {RIPETIZIONE_SCELTA}")
        print("\nProtocolli disponibili:")
        print(df['protocol'].unique())
    else:
        print(f"=== {PROTOCOLLO_SCELTO} / {RIPETIZIONE_SCELTA} ===")
        print(f"Trovate {len(df_filtrato)} ROI\n")
        
        # Imposta opzioni di visualizzazione per vedere tutto
        pd.set_option('display.max_columns', None)
        pd.set_option('display.max_rows', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)
        
        # Mostra tutte le feature per le ROI selezionate
        print(df_filtrato)
        
        print(f"\n✓ Totale feature estratte: {len([col for col in df.columns if col.startswith('original_')])}")


Nessun dato trovato per Abdomen_HD_HR_1.00_Bl60_A0 / rep_1

Protocolli disponibili:
['Abdomen_HD_1,00_Br32_A0' 'Abdomen_HD_1,00_Br32_A3'
 'Abdomen_HD_1,00_Br36_A0' 'Abdomen_HD_1,00_Br36_A3'
 'Abdomen_HD_1,00_Br40_A0' 'Abdomen_HD_1,00_Br40_A3'
 'Abdomen_LD_1,00_Br32_A0' 'Abdomen_LD_1,00_Br32_A3'
 'Abdomen_LD_1,00_Br36_A0' 'Abdomen_LD_1,00_Br36_A3'
 'Abdomen_LD_1,00_Br40_A0' 'Abdomen_LD_1,00_Br40_A3'
 'Abdomen_ST_1,00_Br32_A0' 'Abdomen_ST_1,00_Br32_A3'
 'Abdomen_ST_1,00_Br36_A0' 'Abdomen_ST_1,00_Br36_A3'
 'Abdomen_ST_1,00_Br40_A0' 'Abdomen_ST_1,00_Br40_A3']
